In [ ]:
import sys
import os
from boto3.session import Session


# Get the current notebook's directory
current_dir = os.path.dirname(os.path.abspath('__file__' if '__file__' in globals() else '.'))

utils_dir = os.path.join(current_dir, '..')
utils_dir = os.path.abspath(utils_dir)

# Add to sys.path
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])
from utils import setup_cognito_user_pool, reauthenticate_user

boto_session = Session()
region = boto_session.region_name
print(f"Region: {region}")

In [ ]:
print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool("Cognito_3LO_Github")
print("Cognito setup completed ✓")

In [ ]:
provider_name = "github-provider"

In [ ]:
import os
from bedrock_agentcore.services.identity import IdentityClient
from dotenv import load_dotenv
region = boto_session.region_name
identity_client = IdentityClient(region)

load_dotenv()

github_client_id = os.environ["GITHUB_CLIENT_ID"]
github_client_secret = os.environ["GITHUB_CLIENT_SECRET"]

# Configure GitHub OAuth2 provider - On-Behalf-Of User
github_provider = identity_client.create_oauth2_credential_provider({
    "name": provider_name,
    "credentialProviderVendor": "GithubOauth2",
    "oauth2ProviderConfigInput": {
        "githubOauth2ProviderConfig": {
            'clientId': github_client_id,
            'clientSecret': github_client_secret
        }
    }
})
print(github_provider)
print("\n")
print(f"callbackUrl: {github_provider['callbackUrl']}")

In [ ]:
# Get the OAuth2 callback URL based on the current environment (notebook/SageMaker)
# This is evaluated HERE in the notebook, not in the agent container
from oauth2_callback_server import get_oauth2_callback_url
oauth2_callback_url_for_agent = get_oauth2_callback_url()

print(f"Callback URL for agent (determined from notebook environment): {oauth2_callback_url_for_agent}")

# Write github_agent.py with the callback URL embedded as a string literal
github_agent_code = f'''
import asyncio
import json
import os
from typing import Optional

import httpx
from bedrock_agentcore import BedrockAgentCoreApp
from bedrock_agentcore.identity.auth import requires_access_token
from strands import Agent, tool

# Environment configuration
os.environ["STRANDS_OTEL_ENABLE_CONSOLE_EXPORT"] = "true"
os.environ["OTEL_PYTHON_EXCLUDED_URLS"] = "/ping,/invocations"

# Global token storage (could be improved with a proper state management solution)
github_access_token: Optional[str] = None

app = BedrockAgentCoreApp()


@tool
def inspect_github_repos() -> str:
    """Inspect and list the user's private GitHub repositories.

    Returns:
        str: A JSON string containing the list of repositories and their details,
            or an authentication required message.
    """
    global github_access_token

    if not github_access_token:
        return json.dumps({{
            "auth_required": True,
            "message": "GitHub authentication is required. Please wait while we set up the authorization.",
            "events": []
        }})

    print(f"Using GitHub access token: {{github_access_token[:10]}}...")

    headers = {{"Authorization": f"Bearer {{github_access_token}}"}}

    try:
        with httpx.Client() as client:
            # Get user information
            user_response = client.get("https://api.github.com/user", headers=headers)
            user_response.raise_for_status()
            username = user_response.json().get("login", "Unknown")
            print(f"✅ User: {{username}}")

            # Search for user's repositories
            repos_response = client.get(
                f"https://api.github.com/search/repositories?q=user:{{username}}",
                headers=headers
            )
            repos_response.raise_for_status()
            repos_data = repos_response.json()
            print(f"✅ Found {{len(repos_data.get('items', []))}} repositories")

            repos = repos_data.get('items', [])
            if not repos:
                return f"No repositories found for {{username}}."

            # Format repository information
            response_lines = [f"GitHub repositories for {{username}}:\\n"]

            for repo in repos:
                repo_line = f"📁 {{repo['name']}}"
                if repo.get('language'):
                    repo_line += f" ({{repo['language']}})"
                repo_line += f" - ⭐ {{repo['stargazers_count']}}"
                response_lines.append(repo_line)

                if repo.get('description'):
                    response_lines.append(f"   {{repo['description']}}")
                response_lines.append("")  # Empty line for spacing

            return "\\n".join(response_lines)

    except httpx.HTTPStatusError as e:
        return f"GitHub API error: {{e.response.status_code}} - {{e.response.text}}"
    except Exception as e:
        return f"Error fetching GitHub repositories: {{str(e)}}"


class StreamingQueue:
    """Simple async queue for streaming responses."""

    def __init__(self):
        self._queue = asyncio.Queue()
        self._finished = False

    async def put(self, item: str) -> None:
        """Add an item to the queue."""
        await self._queue.put(item)

    async def finish(self) -> None:
        """Mark the queue as finished and add sentinel value."""
        self._finished = True
        await self._queue.put(None)

    async def stream(self):
        """Stream items from the queue until finished."""
        while True:
            item = await self._queue.get()
            if item is None and self._finished:
                break
            yield item


# Initialize streaming queue
queue = StreamingQueue()


async def on_auth_url(url: str) -> None:
    """Callback for authentication URL."""
    print(f"Authorization URL: {{url}}")
    await queue.put(f"Authorization URL: {{url}}")


def extract_response_text(response) -> str:
    """Extract text content from agent response."""
    if isinstance(response.message, dict):
        content = response.message.get('content', [])
        if isinstance(content, list):
            return "".join(
                item.get('text', '') for item in content
                if isinstance(item, dict) and 'text' in item
            )
    return str(response.message)


def needs_authentication(response_text: str) -> bool:
    """Check if response indicates authentication is required."""
    auth_keywords = [
        "authentication", "authorize", "authorization", "auth",
        "sign in", "login", "access", "permission", "credential",
        "need authentication", "requires authentication"
    ]
    return any(keyword.lower() in response_text.lower() for keyword in auth_keywords)


async def agent_task(user_message: str) -> None:
    """Execute agent task with authentication handling."""
    global github_access_token

    try:
        await queue.put("Begin agent execution")

        # Initial agent call
        response = agent(user_message)
        response_text = extract_response_text(response)

        # Check if authentication is needed
        if needs_authentication(response_text):
            await queue.put("Authentication required for GitHub access. Starting authorization flow...")

            try:
                github_access_token = await need_token_3LO_async(access_token='')
                await queue.put("Authentication successful! Retrying GitHub request...")

                # Retry with authentication
                response = agent(user_message)
            except Exception as auth_error:
                print(f"Authentication error: {{auth_error}}")
                await queue.put(f"Authentication failed: {{str(auth_error)}}")
                return

        await queue.put(response.message)
        await queue.put("End agent execution")

    except Exception as e:
        await queue.put(f"Error: {{str(e)}}")
    finally:
        await queue.finish()


@requires_access_token(
    provider_name="{provider_name}",
    scopes=["repo", "read:user"],
    auth_flow='USER_FEDERATION',
    on_auth_url=on_auth_url,
    force_authentication=False,  # ← Changed to False - will use cached token!
    callback_url="{oauth2_callback_url_for_agent}"  # ← Callback URL determined from notebook environment
)
async def need_token_3LO_async(*, access_token: str) -> str:
    """Handle 3LO authentication flow."""
    global github_access_token
    github_access_token = access_token
    return access_token


# Create agent instance
agent = Agent(
    model="us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    tools=[inspect_github_repos],
    system_prompt="""You are a GitHub assistant. Use the inspect_github_repos tool to fetch private repositories data.
    The inspect_github_repos tool handles token exchange and proper authentication with the GitHub API
    to obtain private information for the user."""
)


@app.entrypoint
async def agent_invocation(payload):
    """Main entrypoint for agent invocation."""
    user_message = payload.get(
        "prompt",
        "No prompt found in input, please guide customer to create a JSON payload with prompt key"
    )

    # Create and start the agent task
    task = asyncio.create_task(agent_task(user_message))

    async def stream_with_task():
        """Stream results while ensuring task completion."""
        async for item in queue.stream():
            yield item
        await task  # Ensure task completes

    return stream_with_task()


if __name__ == "__main__":
    app.run()
'''

# Write the file
with open("github_agent.py", "w", encoding="utf-8") as f:
    f.write(github_agent_code)

print("✅ github_agent.py written successfully with embedded callback URL")

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print(f"Region: {region}")

discovery_url = cognito_config.get("discovery_url")
client_id = cognito_config.get("client_id")

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="github_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_agent_github",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id]
        }
    }
)
response

In [ ]:
!cat .bedrock_agentcore.yaml

In [ ]:
from oauth2_callback_server import get_oauth2_callback_url

# Deploy the agent to AgentCore Runtime and get deployment details
launch_result = agentcore_runtime.launch()
print(launch_result)

if launch_result.agent_id:
    # Extract the workload name from the deployed agent's ID for identity management
    workload_name = launch_result.agent_id
    # Retrieve the current workload identity configuration from AgentCore Identity
    workload_identity = identity_client.get_workload_identity(name=workload_name)
    # Extract existing OAuth2 callback URLs that are already registered for this workload
    allowed_resource_oauth_2_return_urls = workload_identity.get("allowedResourceOauth2ReturnUrls") or []
    # Get the local OAuth2 callback server URL for session binding (localhost:9090/oauth2/callback)
    oauth2_callback_url = get_oauth2_callback_url()
    print(f"Updating workload {workload_name} with callback url {oauth2_callback_url}")

    # Register the local callback URL with the workload identity to enable OAuth2 session binding
    updated_workload_identity = identity_client.update_workload_identity(
        name=workload_name,
        allowed_resource_oauth_2_return_urls=[*allowed_resource_oauth_2_return_urls, oauth2_callback_url],
    )
    print(updated_workload_identity)

In [ ]:
import json
import boto3
agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)

runtime_response = agentcore_control_client.get_agent_runtime(
    agentRuntimeId=launch_result.agent_id
)
runtime_role = runtime_response['roleArn']

policies_to_add = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BedrockModelAccess",
            "Effect": "Allow",
            "Action": [
                "aws-marketplace:ViewSubscriptions",
                "aws-marketplace:Subscribe"
            ],
            "Resource": "*"
        }
    ]
}
iam_client = boto3.client(
    'iam',
    region_name=region
)

response = iam_client.put_role_policy(
    PolicyDocument=json.dumps(policies_to_add),
    PolicyName="outbound_policies",
    RoleName=runtime_role.split("/")[1],
)

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
print(f"Final status: {status}")

In [ ]:
import subprocess
from oauth2_callback_server import store_token_in_oauth2_callback_server, wait_for_oauth2_server_to_be_ready

bearer_token = reauthenticate_user(cognito_config.get("client_id"))

oauth2_callback_server_cmd = [sys.executable, "oauth2_callback_server.py", "--region", region]
oauth2_callback_server_process = subprocess.Popen(oauth2_callback_server_cmd)

try:
    # Start the OAuth2 callback server
    successfully_started_oauth2_server = wait_for_oauth2_server_to_be_ready()
    if not successfully_started_oauth2_server:
        print("Failed to start OAuth2 callback server to handle session binding "
              "(https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html)")
    else:
        store_token_in_oauth2_callback_server(bearer_token)
        invoke_response = agentcore_runtime.invoke(
            {"prompt": "What are my private repositories?"},
            bearer_token=bearer_token
        )
        print(invoke_response)
finally:
    oauth2_callback_server_process.terminate()